<a href="https://colab.research.google.com/github/AmitabhDey-byte/Stock-predictor/blob/main/Stock_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import pandas as pd
import numpy as np
import yfinance as yf
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
df = pd.read_csv('sample_data/stock_news_2016 to 2026.csv')

/tmp/ipykernel_1620/4159371387.py:7: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('sample_data/stock_news_2016 to 2026.csv')


In [24]:
df.head()

,date,title,description,url,source_file,categories,matched_keywords,relevance_score,has_negation,impact_tier
0,2016-01-01,Lending to foreign step-down arms of Indian fi...,RBI slaps 2% additional provision for such loans,URL_NOT_AVAILABLE,IndianFinancialNews.csv,sector_banking_finance,rbi,2.0,False,LOW
1,2016-01-02,IDBI Bank to raise Rs 900 cr through Basel-III...,Many banks have been raising money through tie...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,sector_banking_finance,basel,2.0,False,LOW
2,2016-01-02,Forex reserves up $943 mn,India's foreign exchange reserves rose $943 mi...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,macro_government,forex reserve,3.0,False,MEDIUM
3,2016-01-02,SBI: Lending rate cut unlikely till end-Mar,Country's largest lender SBI today ruled out f...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,stock_specific,sbi,2.0,True,LOW
4,2016-01-03,"ASK Group to invest Rs 1,500 cr in real estate...",ASK Group plans to step up its equity investme...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,sector_cement_infra,real estate,1.0,False,LOW


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76901 entries, 0 to 76900
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   date              76901 non-null  object 
 1   title             76901 non-null  object 
 2   description       76901 non-null  object 
 3   url               76901 non-null  object 
 4   source_file       76900 non-null  object 
 5   categories        76900 non-null  object 
 6   matched_keywords  76900 non-null  object 
 7   relevance_score   76900 non-null  float64
 8   has_negation      76900 non-null  object 
 9   impact_tier       76900 non-null  object 
dtypes: float64(1), object(9)
memory usage: 5.9+ MB


In [26]:
df = df.dropna()
len(df)

76900

In [27]:
#ok so no missing rows

In [28]:
#i need stock market data to merge
#so using yfinance

In [29]:
df2 = yf.download("^NSEI", start="2016-01-01", end="2026-01-01")

/tmp/ipykernel_1620/1582892846.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df2 = yf.download("^NSEI", start="2016-01-01", end="2026-01-01")
[*********************100%***********************]  1 of 1 completed


In [30]:
df2.head()

Price,Close,High,Low,Open,Volume
Ticker,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
Date,,,,,
2016-01-04,7791.299805,7937.549805,7781.100098,7924.549805,134700
2016-01-05,7784.649902,7831.200195,7763.250000,7828.399902,145200
2016-01-06,7741.000000,7800.950195,7721.200195,7788.049805,147100
2016-01-07,7568.299805,7674.950195,7556.600098,7673.350098,188900
2016-01-08,7601.350098,7634.100098,7581.049805,7611.649902,157400


In [31]:
df2.info

<bound method DataFrame.info of Price              Close          High           Low          Open  Volume
Ticker             ^NSEI         ^NSEI         ^NSEI         ^NSEI   ^NSEI
Date                                                                      
2016-01-04   7791.299805   7937.549805   7781.100098   7924.549805  134700
2016-01-05   7784.649902   7831.200195   7763.250000   7828.399902  145200
2016-01-06   7741.000000   7800.950195   7721.200195   7788.049805  147100
2016-01-07   7568.299805   7674.950195   7556.600098   7673.350098  188900
2016-01-08   7601.350098   7634.100098   7581.049805   7611.649902  157400
...                  ...           ...           ...           ...     ...
2025-12-24  26142.099609  26236.400391  26123.000000  26170.650391  188800
2025-12-26  26042.300781  26144.199219  26008.599609  26121.250000  142200
2025-12-29  25942.099609  26106.800781  25920.300781  26063.349609  234300
2025-12-30  25938.849609  25976.750000  25878.000000  25940.900391  396900
2025-12-31  26129.599609  26187.949219  25969.000000  25971.050781  246300

[2464 rows x 5 columns]>

In [32]:
df['date'] = pd.to_datetime(df['date']).dt.date

In [33]:
df['news_data'] = df['title'].astype(str) + " " + df['description'].astype(str)

print(df['news_data'].head(1))

0    Lending to foreign step-down arms of Indian fi...
Name: news_data, dtype: object


In [34]:
df2.columns = [col[0] if isinstance(col,  tuple) else col for col in df2.columns]

In [35]:
df2.reset_index(inplace=True)
df2.rename(columns={"Price Ticker Date": "date"}, inplace=True)
df2['date'] = pd.to_datetime(df['date']).dt.date

In [36]:
df2.head()

,Date,Close,High,Low,Open,Volume,date
0,2016-01-04,7791.299805,7937.549805,7781.100098,7924.549805,134700,2016-01-01
1,2016-01-05,7784.649902,7831.200195,7763.250000,7828.399902,145200,2016-01-02
2,2016-01-06,7741.000000,7800.950195,7721.200195,7788.049805,147100,2016-01-02
3,2016-01-07,7568.299805,7674.950195,7556.600098,7673.350098,188900,2016-01-02
4,2016-01-08,7601.350098,7634.100098,7581.049805,7611.649902,157400,2016-01-03


In [37]:
df3 = pd.merge(df, df2, on='date')

In [38]:
df3.head(6)

,date,title,description,url,source_file,categories,matched_keywords,relevance_score,has_negation,impact_tier,news_data,Date,Close,High,Low,Open,Volume
0,2016-01-01,Lending to foreign step-down arms of Indian fi...,RBI slaps 2% additional provision for such loans,URL_NOT_AVAILABLE,IndianFinancialNews.csv,sector_banking_finance,rbi,2.0,False,LOW,Lending to foreign step-down arms of Indian fi...,2016-01-04,7791.299805,7937.549805,7781.100098,7924.549805,134700
1,2016-01-02,IDBI Bank to raise Rs 900 cr through Basel-III...,Many banks have been raising money through tie...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,sector_banking_finance,basel,2.0,False,LOW,IDBI Bank to raise Rs 900 cr through Basel-III...,2016-01-05,7784.649902,7831.200195,7763.250000,7828.399902,145200
2,2016-01-02,IDBI Bank to raise Rs 900 cr through Basel-III...,Many banks have been raising money through tie...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,sector_banking_finance,basel,2.0,False,LOW,IDBI Bank to raise Rs 900 cr through Basel-III...,2016-01-06,7741.000000,7800.950195,7721.200195,7788.049805,147100
3,2016-01-02,IDBI Bank to raise Rs 900 cr through Basel-III...,Many banks have been raising money through tie...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,sector_banking_finance,basel,2.0,False,LOW,IDBI Bank to raise Rs 900 cr through Basel-III...,2016-01-07,7568.299805,7674.950195,7556.600098,7673.350098,188900
4,2016-01-02,Forex reserves up $943 mn,India's foreign exchange reserves rose $943 mi...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,macro_government,forex reserve,3.0,False,MEDIUM,Forex reserves up $943 mn India's foreign exch...,2016-01-05,7784.649902,7831.200195,7763.250000,7828.399902,145200
5,2016-01-02,Forex reserves up $943 mn,India's foreign exchange reserves rose $943 mi...,URL_NOT_AVAILABLE,IndianFinancialNews.csv,macro_government,forex reserve,3.0,False,MEDIUM,Forex reserves up $943 mn India's foreign exch...,2016-01-06,7741.000000,7800.950195,7721.200195,7788.049805,147100


In [39]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64886 entries, 0 to 64885
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              64886 non-null  object        
 1   title             64886 non-null  object        
 2   description       64886 non-null  object        
 3   url               64886 non-null  object        
 4   source_file       64886 non-null  object        
 5   categories        64886 non-null  object        
 6   matched_keywords  64886 non-null  object        
 7   relevance_score   64886 non-null  float64       
 8   has_negation      64886 non-null  object        
 9   impact_tier       64886 non-null  object        
 10  news_data         64886 non-null  object        
 11  Date              64886 non-null  datetime64[ns]
 12  Close             64886 non-null  float64       
 13  High              64886 non-null  float64       
 14  Low               6488

In [40]:
df3 = df3.drop(['date','title','description','url','source_file'], axis=1)

In [41]:
df3.head()

,categories,matched_keywords,relevance_score,has_negation,impact_tier,news_data,Date,Close,High,Low,Open,Volume
0,sector_banking_finance,rbi,2.0,False,LOW,Lending to foreign step-down arms of Indian fi...,2016-01-04,7791.299805,7937.549805,7781.100098,7924.549805,134700
1,sector_banking_finance,basel,2.0,False,LOW,IDBI Bank to raise Rs 900 cr through Basel-III...,2016-01-05,7784.649902,7831.200195,7763.250000,7828.399902,145200
2,sector_banking_finance,basel,2.0,False,LOW,IDBI Bank to raise Rs 900 cr through Basel-III...,2016-01-06,7741.000000,7800.950195,7721.200195,7788.049805,147100
3,sector_banking_finance,basel,2.0,False,LOW,IDBI Bank to raise Rs 900 cr through Basel-III...,2016-01-07,7568.299805,7674.950195,7556.600098,7673.350098,188900
4,macro_government,forex reserve,3.0,False,MEDIUM,Forex reserves up $943 mn India's foreign exch...,2016-01-05,7784.649902,7831.200195,7763.250000,7828.399902,145200


In [42]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64886 entries, 0 to 64885
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   categories        64886 non-null  object        
 1   matched_keywords  64886 non-null  object        
 2   relevance_score   64886 non-null  float64       
 3   has_negation      64886 non-null  object        
 4   impact_tier       64886 non-null  object        
 5   news_data         64886 non-null  object        
 6   Date              64886 non-null  datetime64[ns]
 7   Close             64886 non-null  float64       
 8   High              64886 non-null  float64       
 9   Low               64886 non-null  float64       
 10  Open              64886 non-null  float64       
 11  Volume            64886 non-null  int64         
dtypes: datetime64[ns](1), float64(5), int64(1), object(5)
memory usage: 5.9+ MB


In [43]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

In [44]:
df3['sentiment'] = df3['news_data'].apply(
    lambda x: analyzer.polarity_scores(str(x))['compound']
)

In [45]:
df3[['news_data', 'sentiment']].head(20)

,news_data,sentiment
0,Lending to foreign step-down arms of Indian fi...,0.0000
1,IDBI Bank to raise Rs 900 cr through Basel-III...,0.0000
2,IDBI Bank to raise Rs 900 cr through Basel-III...,0.0000
3,IDBI Bank to raise Rs 900 cr through Basel-III...,0.0000
4,Forex reserves up $943 mn India's foreign exch...,0.0000
5,Forex reserves up $943 mn India's foreign exch...,0.0000
6,Forex reserves up $943 mn India's foreign exch...,0.0000
7,SBI: Lending rate cut unlikely till end-Mar Co...,-0.4939
8,SBI: Lending rate cut unlikely till end-Mar Co...,-0.4939
9,SBI: Lending rate cut unlikely till end-Mar Co...,-0.4939


In [46]:
# 5 columns removed which were unnecessary to remove noise

In [47]:
df3['target'] = df3['Close'].shift(-1)

In [48]:
df3= df3.dropna()

In [49]:
X = df3[['Close','High','Low','Open','Volume','sentiment']]
Y =df3['target']

In [50]:
X_train, X_test, y_train, y_test = train_test_split(X, Y,test_size=0.2,shuffle=False)

In [51]:
model = XGBRegressor(n_estimators=300,learning_rate=0.05,max_depth=4,
                      subsample=0.8,colsample_bytree=0.8,
                       reg_alpha=1,reg_lambda=1,
                     tree_method='hist',device='cuda')

model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [52]:
model.get_params()

{'objective': 'reg:squarederror',
 'base_score': None,
 'booster': None,
 'callbacks': None,
 'colsample_bylevel': None,
 'colsample_bynode': None,
 'colsample_bytree': 0.8,
 'device': 'cuda',
 'early_stopping_rounds': None,
 'enable_categorical': False,
 'eval_metric': None,
 'feature_types': None,
 'feature_weights': None,
 'gamma': None,
 'grow_policy': None,
 'importance_type': None,
 'interaction_constraints': None,
 'learning_rate': 0.05,
 'max_bin': None,
 'max_cat_threshold': None,
 'max_cat_to_onehot': None,
 'max_delta_step': None,
 'max_depth': 4,
 'max_leaves': None,
 'min_child_weight': None,
 'missing': nan,
 'monotone_constraints': None,
 'multi_strategy': None,
 'n_estimators': 300,
 'n_jobs': None,
 'num_parallel_tree': None,
 'random_state': None,
 'reg_alpha': 1,
 'reg_lambda': 1,
 'sampling_method': None,
 'scale_pos_weight': None,
 'subsample': 0.8,
 'tree_method': 'hist',
 'validate_parameters': None,
 'verbosity': None}

In [53]:
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [10:38:46] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [54]:
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)

train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
print("Train R2:", train_r2)
print("Test R2 :", test_r2)
print("Train RMSE:", train_rmse)
print("Test RMSE :", test_rmse)

Train R2: 0.9989992506513778
Test R2 : 0.9237272489318917
Train RMSE: 142.093739804397
Test RMSE : 249.44690713109165


there is a bit or mild overlifting....but fixable..all left to do is apply sentimen analysis and then grid search

In [55]:
estimators_list = [100,200, 300,400, 500]

for n in estimators_list:

    model = XGBRegressor(n_estimators=n,max_depth=4,learning_rate=0.03,tree_method='hist',device='cuda')
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)
    print(f"{n}...Train R2: {train_r2*100}%")
    print(f"{n}...Test R2 : {test_r2*100}%")

100...Train R2: 99.56272667422427%
100...Test R2 : 65.071291743011%
200...Train R2: 99.8447088588053%
200...Test R2 : 89.93830127068028%
300...Train R2: 99.87107373155179%
300...Test R2 : 92.70299908922057%
400...Train R2: 99.88838131989388%
400...Test R2 : 92.36369390410485%
500...Train R2: 99.90113717924152%
500...Test R2 : 92.33618784876555%


i think keeping n_estimators at 300 will be good..there is almost no overlifting

In [56]:
max_depth_list = [1,2,3,4,5,6]

for n in max_depth_list:

    model = XGBRegressor(n_estimators=300,max_depth=n,learning_rate=0.03,tree_method='hist',device='cuda')
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)
    print(f"{n}...Train R2: {train_r2*100}%")
    print(f"{n}...Test R2 : {test_r2*100}%")

1...Train R2: 99.7236032858999%
1...Test R2 : 70.6217106754794%
2...Train R2: 99.79330996139203%
2...Test R2 : 90.91731018015234%
3...Train R2: 99.82639532912754%
3...Test R2 : 92.51748025600918%
4...Train R2: 99.87107373155179%
4...Test R2 : 92.70299908922057%
5...Train R2: 99.91162409487464%
5...Test R2 : 93.50144512346044%
6...Train R2: 99.94497008245499%
6...Test R2 : 92.65451814780613%


In [57]:
#keeping max depth at 5 and n estimators at 300

In [58]:
learning_list = [0.01, 0.02,0.03,0.05,0.07,0.09,0.1]

for n in learning_list:

    model = XGBRegressor(n_estimators=300,max_depth=5,learning_rate=n,tree_method='hist',device='cuda')
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)
    print(f"{n}...Train R2: {train_r2*100}%")
    print(f"{n}...Test R2 : {test_r2*100}%")

0.01...Train R2: 99.58751698512056%
0.01...Test R2 : 66.79831831917889%
0.02...Train R2: 99.88632123221377%
0.02...Test R2 : 92.3626197922192%
0.03...Train R2: 99.91162409487464%
0.03...Test R2 : 93.50144512346044%
0.05...Train R2: 99.93751212007939%
0.05...Test R2 : 92.14096271604471%
0.07...Train R2: 99.95454005733286%
0.07...Test R2 : 91.87302686395844%
0.09...Train R2: 99.96603841769186%
0.09...Test R2 : 92.19751202263596%
0.1...Train R2: 99.97022441853454%
0.1...Test R2 : 93.19850971339294%


In [59]:
#then n estimators: 300, max depth: 5, learning rate: 0.03

In [60]:
subsample_list = [0.5,0.6,0.8,1.0]

for n in subsample_list:

    model = XGBRegressor(n_estimators=300,max_depth=5,learning_rate=0.03,subsample=n,tree_method='hist',device='cuda')
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)
    print(f"{n}...Train R2: {train_r2*100}%")
    print(f"{n}...Test R2 : {test_r2*100}%")

0.5...Train R2: 99.91193771472828%
0.5...Test R2 : 93.82844371768854%
0.6...Train R2: 99.91160250077726%
0.6...Test R2 : 93.66682093071972%
0.8...Train R2: 99.91127632547415%
0.8...Test R2 : 93.43840099024709%
1.0...Train R2: 99.91162409487464%
1.0...Test R2 : 93.50144512346044%


In [61]:
#then n estimators: 300, max depth: 5, learning rate: 0.03, subsample=0.5

In [62]:
from sklearn.model_selection import GridSearchCV

In [63]:
param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [4, 5],
    'learning_rate': [0.03, 0.05],
    'subsample': [0.5, 0.8],
    'colsample_bytree': [0.7, 0.8],
    'reg_alpha': [0.1, 1],
    'reg_lambda': [1],
}

In [64]:
xgb = XGBRegressor(
    tree_method='hist',
    device='cuda'
)

In [65]:
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    verbose=1,
    n_jobs=-1
)

In [66]:
grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 64 candidates, totalling 192 fits


GridSearchCV(cv=3,
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device='cuda',
                                    early_stopping_rounds=None,
                                    enable_categorical=False, eval_metric=None,
                                    feature_types=None, feature_weights=None,
                                    gamma=None, grow_policy=None,
                                    importance_type=None,
                                    interaction_constraints=No...
                                    max_depth=None, max_leaves=None,
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None,
                                    multi_strategy=None, n_estimators=None,
                                    n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.7, 0.8],
                         'learning_rate': [0.03, 0.05], 'max_depth': [4, 5],
                         'n_estimators': [200, 300], 'reg_alpha': [0.1, 1],
                         'reg_lambda': [1], 'subsample': [0.5, 0.8]},
             scoring='r2', verbose=1)

In [67]:
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)
mae = mean_absolute_error(y_test, test_pred)
mse = mean_squared_error(y_test, test_pred)
rmse = mse ** 0.5
print(f"Train R2 Score : {train_r2}")
print(f"Test R2 Score  : {test_r2}")
print(f"MAE            : {mae}")
print(f"MSE            : {mse}")
print(f"RMSE           : {rmse}")

Train R2 Score : 0.9991162409487464
Test R2 Score  : 0.9350144512346045
MAE            : 165.66989327873353
MSE            : 53015.593370500086
RMSE           : 230.25115281036074


In [68]:
import joblib

joblib.dump(model, 'xgboost_stock_model.pkl')

print("Model Saved Successfully!")

Model Saved Successfully!


In [69]:
import joblib

loaded_model = joblib.load('xgboost_stock_model.pkl')

print("Model Loaded Successfully!")

Model Loaded Successfully!


In [70]:
prediction = loaded_model.predict(X_test)

print(prediction[:5])

[24797.326 24636.05  24631.63  24751.586 24888.664]
